# Potency level labeling

1. Edit **INPUT_CSV** and **OUTPUT_CSV** in the next cell
2. Run all cells

Output CSV has `potency_level` (Weak/Moderate/Strong).

### Project paths

This notebook lives in `notebooks/`. Shared code is under `src/`, data under `data/`.
On Colab, set `ROOT` to your cloned repo (or Drive copy) instead of `Path('..')`.


In [ ]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/). On Colab, replace with your clone path, e.g.:
# ROOT = Path('/content/drive/MyDrive/your-thesis-repo')
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.paths import DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, PROJECT_ROOT
print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('CHECKPOINTS  :', CHECKPOINTS_DIR)
print('RESULTS      :', RESULTS_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import sys

INPUT_PATH = "/content/drive/MyDrive/DBAASP_Bioinformatics"
os.chdir(INPUT_PATH)
sys.path.append(INPUT_PATH)
print("Working dir:", os.getcwd())


In [ ]:
INPUT_CSV  = INPUT_PATH + "/databases/peptide_organism_mic.csv"
OUTPUT_CSV = INPUT_PATH + "/databases/diffusion_training_ready.csv"
N_LEVELS   = 3
WANDB_PROJECT = "amp-diffusion"
KEEP_ORGANISMS = None

print("Input :", INPUT_CSV)
print("Output:", OUTPUT_CSV)


In [ ]:
!pip install -q wandb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import wandb
import os

LEVEL_SCHEMES = {
    3: {
        "labels": ["Weak", "Moderate", "Strong"],
        "potency_bins": [0, 33.33, 66.67, 100.01],
        "cut_pcts": [33.33, 66.67],
    },
    5: {
        "labels": ["Very Weak", "Weak", "Moderate", "Strong", "Very Strong"],
        "potency_bins": [0, 20, 40, 60, 80, 100.01],
        "cut_pcts": [20, 40, 60, 80],
    },
}


def pick_column(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Missing {label}. Tried {candidates}. Have: {list(df.columns)}")


def scheme_for(n_levels):
    if n_levels not in LEVEL_SCHEMES:
        raise ValueError("N_LEVELS must be 3 or 5")
    return LEVEL_SCHEMES[n_levels]


def compute_levels(df, n_levels=3):
    sch = scheme_for(n_levels)
    df = df.copy()
    df["mic_ug_ml"] = df["mic_ug_ml"].clip(lower=1e-4)
    df["log10_mic"] = np.log10(df["mic_ug_ml"])

    parts = []
    for organism, g in df.groupby("organism"):
        g = g.copy()
        pct_rank = g["log10_mic"].rank(pct=True)
        g["potency_percentile"] = (100 * (1 - pct_rank)).round(1)
        mu, sd = g["log10_mic"].mean(), g["log10_mic"].std()
        if sd == 0:
            sd = 1.0
        g["potency_zscore"] = ((mu - g["log10_mic"]) / sd).round(3)
        g["potency_level"] = pd.cut(
            g["potency_percentile"],
            bins=[-0.01] + sch["potency_bins"][1:],
            labels=sch["labels"],
        )
        parts.append(g)
    return pd.concat(parts, ignore_index=True)


def organism_cutoffs(g, n_levels):
    """MIC thresholds on log10 scale where potency percentile bins change."""
    sch = scheme_for(n_levels)
    log_mic = g["log10_mic"].values
    mic = g["mic_ug_ml"].values
    n = len(g)
    rows = []
    for cp in sch["cut_pcts"]:
        log_thr = float(np.percentile(log_mic, 100 - cp))
        mic_thr = float(10 ** log_thr)
        rows.append({
            "potency_percentile_cutoff": cp,
            "log10_mic_threshold": round(log_thr, 4),
            "mic_ug_ml_threshold": round(mic_thr, 4),
            "rule": f"potency_percentile {'>' if cp < 50 else '>='} {cp} ↔ log10(MIC) {'≤' if cp >= 50 else '≥'} {log_thr:.3f}",
        })
    return pd.DataFrame(rows), {
        "n_total": n,
        "mic_min": round(float(mic.min()), 4),
        "mic_median": round(float(np.median(mic)), 4),
        "mic_max": round(float(mic.max()), 4),
        "log10_mic_min": round(float(log_mic.min()), 4),
        "log10_mic_median": round(float(np.median(log_mic)), 4),
        "log10_mic_max": round(float(log_mic.max()), 4),
    }


def class_definition_table(df, n_levels):
    """One row per organism × class with counts, percentile range, MIC range."""
    sch = scheme_for(n_levels)
    labels = sch["labels"]
    bins = sch["potency_bins"]
    rows = []

    for organism, g in df.groupby("organism"):
        n_org = len(g)
        cut_df, stats = organism_cutoffs(g, n_levels)

        for i, class_name in enumerate(labels):
            mask = g["potency_level"] == class_name
            n_class = int(mask.sum())
            pct_class = round(100 * n_class / n_org, 1) if n_org else 0.0
            pot_min = bins[i]
            pot_max = bins[i + 1] if bins[i + 1] <= 100 else 100.0

            sub = g.loc[mask, "mic_ug_ml"]
            if n_class > 0:
                mic_lo = round(float(sub.min()), 4)
                mic_hi = round(float(sub.max()), 4)
                log_lo = round(float(np.log10(sub.min())), 4)
                log_hi = round(float(np.log10(sub.max())), 4)
            else:
                mic_lo = mic_hi = log_lo = log_hi = None

            if i == 0:
                boundary_hi = cut_df.iloc[0]["mic_ug_ml_threshold"] if len(cut_df) else None
                mic_rule = f"MIC ≥ {boundary_hi} ug/mL (weakest third)" if n_levels == 3 else f"lowest potency bin"
            elif i == len(labels) - 1:
                boundary_lo = cut_df.iloc[-1]["mic_ug_ml_threshold"] if len(cut_df) else None
                mic_rule = f"MIC ≤ {boundary_lo} ug/mL (most potent third)" if n_levels == 3 else f"highest potency bin"
            else:
                lo = cut_df.iloc[i - 1]["mic_ug_ml_threshold"]
                hi = cut_df.iloc[i]["mic_ug_ml_threshold"]
                mic_rule = f"{lo} < MIC ≤ {hi} ug/mL"

            rows.append({
                "organism": organism,
                "organism_n_total": n_org,
                "class_name": class_name,
                "class_n": n_class,
                "class_pct_of_organism": pct_class,
                "potency_percentile_min": pot_min,
                "potency_percentile_max": pot_max,
                "potency_percentile_rule": f"{pot_min} < potency_percentile ≤ {pot_max}",
                "mic_min_ug_ml_in_class": mic_lo,
                "mic_max_ug_ml_in_class": mic_hi,
                "log10_mic_min_in_class": log_lo,
                "log10_mic_max_in_class": log_hi,
                "mic_assignment_rule": mic_rule,
            })

    return pd.DataFrame(rows)


def organism_summary_table(df, n_levels):
    """One row per organism: dataset size + cutoff MIC values."""
    sch = scheme_for(n_levels)
    rows = []
    for organism, g in df.groupby("organism"):
        cut_df, stats = organism_cutoffs(g, n_levels)
        row = {
            "organism": organism,
            "n_peptides": stats["n_total"],
            "mic_min_ug_ml": stats["mic_min"],
            "mic_median_ug_ml": stats["mic_median"],
            "mic_max_ug_ml": stats["mic_max"],
            "labeling_method": "organism-relative tertiles on log10(MIC)" if n_levels == 3 else f"{n_levels}-level organism-relative bins",
            "class_names": ", ".join(sch["labels"]),
        }
        for _, cr in cut_df.iterrows():
            cp = cr["potency_percentile_cutoff"]
            row[f"cutoff_potency_pct_{cp}"] = cp
            row[f"cutoff_mic_ug_ml_at_pct_{cp}"] = cr["mic_ug_ml_threshold"]
            row[f"cutoff_log10_mic_at_pct_{cp}"] = cr["log10_mic_threshold"]
        for lbl in sch["labels"]:
            row[f"n_{lbl.replace(' ', '_')}"] = int((g["potency_level"] == lbl).sum())
        rows.append(row)
    return pd.DataFrame(rows)


CLASS_COLORS = {
    "Very Weak": "#d62728",
    "Weak": "#ff7f0e",
    "Moderate": "#bcbd22",
    "Strong": "#2ca02c",
    "Very Strong": "#1f77b4",
}


def format_mic(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "—"
    if v >= 100:
        return f"{v:.0f}"
    if v >= 10:
        return f"{v:.1f}"
    if v >= 1:
        return f"{v:.2f}"
    return f"{v:.3g}"


def organism_class_summary(g, organism, n_levels):
    """Clean per-class rows for display tables."""
    sch = scheme_for(n_levels)
    labels = sch["labels"]
    bins = sch["potency_bins"]
    n = len(g)
    log_mic = g["log10_mic"].values
    cut_pcts = sch["cut_pcts"]
    mic_cuts = [10 ** float(np.percentile(log_mic, 100 - cp)) for cp in cut_pcts]

    rows = []
    for i, class_name in enumerate(labels):
        sub = g[g["potency_level"] == class_name]
        nc = len(sub)
        pct = round(100 * nc / n, 1) if n else 0.0
        pot_lo, pot_hi = bins[i], bins[i + 1]
        pot_range = f"{pot_lo}–{pot_hi}" if pot_hi <= 100 else f">{pot_lo}"

        if nc:
            mic_obs = f"{format_mic(sub['mic_ug_ml'].min())} – {format_mic(sub['mic_ug_ml'].max())}"
        else:
            mic_obs = "—"

        if i == len(labels) - 1:
            mic_rule = f"MIC ≤ {format_mic(mic_cuts[-1])} ug/mL"
        elif i == 0:
            mic_rule = f"MIC > {format_mic(mic_cuts[0])} ug/mL"
        else:
            mic_rule = f"{format_mic(mic_cuts[i-1])} < MIC ≤ {format_mic(mic_cuts[i])} ug/mL"

        rows.append({
            "class": class_name,
            "peptides": nc,
            "pct": pct,
            "potency_percentile": pot_range,
            "mic_threshold": mic_rule,
            "mic_observed_range": mic_obs,
        })
    return pd.DataFrame(rows)


def plot_organism_histogram(g, organism, n_levels):
    """Full-width MIC histogram with shaded classes."""
    sch = scheme_for(n_levels)
    labels = sch["labels"]
    cut_pcts = sch["cut_pcts"]
    log_mic = g["log10_mic"].values
    n = len(g)

    log_cuts = [float(np.percentile(log_mic, 100 - cp)) for cp in cut_pcts]
    mic_cuts = [10 ** x for x in log_cuts]

    fig, ax = plt.subplots(figsize=(12, 4.5))
    xmin, xmax = log_mic.min() - 0.05, log_mic.max() + 0.05
    edges = [xmin] + log_cuts + [xmax]

    for i, class_name in enumerate(reversed(labels)):
        idx = len(labels) - 1 - i
        ax.axvspan(edges[idx], edges[idx + 1], alpha=0.30, color=CLASS_COLORS.get(class_name, "#ccc"))

    ax.hist(log_mic, bins=min(40, max(12, n // 12)), color="#2f2f2f", alpha=0.50, edgecolor="white")

    for cp, lc, mc in zip(cut_pcts, log_cuts, mic_cuts):
        ax.axvline(lc, color="#c0392b", linestyle="--", linewidth=2)
        ax.annotate(
            f"percentile {cp}\nMIC = {format_mic(mc)} ug/mL",
            xy=(lc, ax.get_ylim()[1]), xytext=(8, -8), textcoords="offset points",
            fontsize=9, color="#c0392b", va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#c0392b", alpha=0.9),
        )

    patches = [mpatches.Patch(color=CLASS_COLORS.get(l, "#ccc"), label=l) for l in labels]
    ax.legend(handles=patches, loc="upper right", title="Class", framealpha=0.95)
    ax.set_xlabel("log10(MIC, ug/mL)   ← lower = more potent", fontsize=11)
    ax.set_ylabel("Peptide count", fontsize=11)
    ax.set_title(f"{organism}  ·  {n} peptides  ·  organism-relative {n_levels}-level labeling", fontsize=13, pad=10)
    plt.tight_layout()
    return fig


def plot_organism_class_table(summary_df, organism, n_levels, mic_cuts, cut_pcts):
    """Full-width readable class summary table (separate image)."""
    sch = scheme_for(n_levels)
    n = int(summary_df["peptides"].sum())

    fig, ax = plt.subplots(figsize=(12, 2.2 + divmod(len(summary_df), 1)[0] * 0.55))
    ax.axis("off")

    headers = [
        "Class",
        "Peptides",
        "% of organism",
        "Potency percentile",
        "MIC threshold (assignment rule)",
        "MIC range (observed in data)",
    ]
    body = []
    for _, row in summary_df.iterrows():
        body.append([
            row["class"],
            str(int(row["peptides"])),
            f"{row['pct']:.1f}%",
            row["potency_percentile"],
            row["mic_threshold"],
            row["mic_observed_range"],
        ])

    cutoff_lines = [
        f"Organism total: {n} peptides  |  Labeling: tertiles within this organism on log10(MIC)",
        "Cutoff MIC thresholds:  "
        + "   ·   ".join(
            f"percentile {cp} → {format_mic(mc)} ug/mL" for cp, mc in zip(cut_pcts, mic_cuts)
        ),
    ]

    table = ax.table(
        cellText=body,
        colLabels=headers,
        loc="center",
        cellLoc="left",
        colLoc="left",
        colWidths=[0.11, 0.09, 0.11, 0.14, 0.28, 0.27],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2.0)

    for (r, c), cell in table.get_celld().items():
        if r == 0:
            cell.set_facecolor("#2c5282")
            cell.set_text_props(color="white", weight="bold", fontsize=11)
            cell.set_height(0.12)
        else:
            class_name = body[r - 1][0]
            if c == 0:
                cell.set_facecolor(CLASS_COLORS.get(class_name, "#eee"))
                cell.set_text_props(weight="bold")
            else:
                cell.set_facecolor("#f8f9fa")
            cell.set_height(0.14)

    fig.suptitle(f"{organism} — per-class summary", fontsize=14, weight="bold", y=0.98)
    fig.text(0.5, 0.02, cutoff_lines[0] + "\n" + cutoff_lines[1], ha="center", va="bottom", fontsize=9, color="#444")
    plt.subplots_adjust(top=0.82, bottom=0.14)
    return fig


def plot_organism_panel(g, organism, n_levels):
    """Returns (histogram_fig, table_fig) for cleaner W&B logging."""
    sch = scheme_for(n_levels)
    log_mic = g["log10_mic"].values
    cut_pcts = sch["cut_pcts"]
    mic_cuts = [10 ** float(np.percentile(log_mic, 100 - cp)) for cp in cut_pcts]
    summary = organism_class_summary(g, organism, n_levels)
    fig_hist = plot_organism_histogram(g, organism, n_levels)
    fig_tbl = plot_organism_class_table(summary, organism, n_levels, mic_cuts, cut_pcts)
    return fig_hist, fig_tbl, summary


raw = pd.read_csv(INPUT_CSV)
if raw.shape[1] == 1:
    raw = pd.read_csv(INPUT_CSV, sep="\t")
print(f"Loaded {len(raw)} rows: {list(raw.columns)}")

seq_c = pick_column(raw, ["sequence", "peptide", "Sequence"], "sequence")
org_c = pick_column(raw, ["organism", "Organism"], "organism")
mic_c = pick_column(raw, ["mic_ug_ml", "MIC", "mic", "MIC_ug_ml"], "MIC")

df = pd.DataFrame({
    "sequence": raw[seq_c].astype(str).str.upper().str.strip(),
    "organism": raw[org_c].astype(str).str.strip(),
    "mic_ug_ml": pd.to_numeric(raw[mic_c], errors="coerce"),
})
df = df.dropna(subset=["mic_ug_ml"])
df = df[df["sequence"].str.len() > 0]

if KEEP_ORGANISMS is not None:
    df = df[df["organism"].isin(KEEP_ORGANISMS)]
    print("Filtered organisms:", KEEP_ORGANISMS)

print(f"Rows after clean: {len(df)}")

df = compute_levels(df, n_levels=N_LEVELS)
class_defs = class_definition_table(df, N_LEVELS)
org_summary = organism_summary_table(df, N_LEVELS)
counts = pd.crosstab(df["organism"], df["potency_level"])

print("\nOrganism summary:")
print(org_summary.to_string(index=False))
print("\nClass definitions (first 15 rows):")
print(class_defs.head(15).to_string(index=False))


out = df[["sequence", "organism", "potency_percentile", "potency_zscore", "potency_level"]]
os.makedirs(os.path.dirname(os.path.abspath(OUTPUT_CSV)), exist_ok=True)
out.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(out)} rows -> {OUTPUT_CSV}")


wandb.login()
sch = scheme_for(N_LEVELS)
wandb.init(
    project=WANDB_PROJECT,
    job_type="potency-leveling",
    config={
        "input_csv": INPUT_CSV,
        "output_csv": OUTPUT_CSV,
        "n_levels": N_LEVELS,
        "class_names": sch["labels"],
        "n_rows": len(df),
        "n_organisms": int(df["organism"].nunique()),
        "labeling_method": "Within each organism: rank by log10(MIC), invert to potency_percentile (100=most potent), assign tertile class",
        "potency_percentile_cutoffs": sch["cut_pcts"],
        "note": "MIC thresholds differ per organism because labeling is organism-relative",
    },
)

wandb.log({
    "summary/organism_overview": wandb.Table(dataframe=org_summary),
    "summary/class_definitions_per_organism": wandb.Table(dataframe=class_defs),
    "summary/organism_x_class_counts": wandb.Table(dataframe=counts.reset_index()),
    "summary/sample_labeled_peptides": wandb.Table(dataframe=df.head(300)),
})

for organism, g in df.groupby("organism"):
    safe = organism.replace("/", "-").replace(" ", "_")
    fig_hist, fig_tbl, summary = plot_organism_panel(g, organism, N_LEVELS)
    wandb.log({
        f"by_organism/{safe}/01_histogram": wandb.Image(fig_hist),
        f"by_organism/{safe}/02_class_summary_table": wandb.Image(fig_tbl),
        f"by_organism/{safe}/03_class_summary_data": wandb.Table(dataframe=summary),
    })
    plt.close(fig_hist)
    plt.close(fig_tbl)

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(counts))
x = np.arange(len(counts.index))
for class_name in sch["labels"]:
    if class_name in counts.columns:
        vals = counts[class_name].values
        color = CLASS_COLORS.get(class_name, None)
        ax.bar(x, vals, bottom=bottom, label=class_name, color=color)
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 0:
                ax.text(i, b + v / 2, str(int(v)), ha="center", va="center", fontsize=7, color="white", weight="bold")
        bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(counts.index, rotation=45, ha="right")
ax.set_ylabel("peptide count")
ax.set_title("Data quantity per organism × potency class")
ax.legend(title="Class name", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
wandb.log({"summary/organism_class_stacked_counts": wandb.Image(fig)})
plt.close(fig)

art = wandb.Artifact("diffusion_training_ready", type="dataset")
art.add_file(OUTPUT_CSV)
class_defs.to_csv(OUTPUT_CSV.replace(".csv", "_class_definitions.csv"), index=False)
org_summary.to_csv(OUTPUT_CSV.replace(".csv", "_organism_summary.csv"), index=False)
art.add_file(OUTPUT_CSV.replace(".csv", "_class_definitions.csv"))
art.add_file(OUTPUT_CSV.replace(".csv", "_organism_summary.csv"))
wandb.log_artifact(art)

wandb.finish()
print("Done. W&B logged with per-organism panels and class definition tables.")
print("Next: encode_latents.ipynb on", OUTPUT_CSV)
